In [ ]:
import mne
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt

In [ ]:
file_path = "D:/Users/windows/Desktop/Inner Speech Decoding/Dataset/sub-01_ses-01_eeg-epo.fif"
epochs = mne.read_epochs(file_path, preload=True)
print(epochs)
print("Event IDs:", epochs.event_id)

In [ ]:
label_map = {31: 'up', 32: 'down', 33: 'right', 34: 'left'}
classes = sorted(label_map.keys())
sfreq = epochs.info['sfreq']
data = epochs.get_data(copy=True)           # (n_epochs, n_channels, n_times)
labels = epochs.events[:, 2]               # integer class codes
n_epochs, n_channels, n_times = data.shape
print(f"Data shape: {data.shape}, sfreq: {sfreq} Hz")

In [ ]:
# --- Filter bank definition: 30 bands from 0-60 Hz at 2 Hz intervals ---
# Each band: [f_low, f_high] with 2 Hz width; first band starts at 1 Hz to avoid DC
band_edges = [(max(1, f), f + 2) for f in range(0, 60, 2)]   # 30 bands
print(f"Number of bands: {len(band_edges)}")
print(f"Bands: {band_edges[:3]} ... {band_edges[-3:]}")

def bandpass(signal, lo, hi, fs, order=4):
    """Apply zero-phase Butterworth bandpass filter."""
    nyq = fs / 2.0
    lo_n, hi_n = lo / nyq, hi / nyq
    # clamp to (0, 1) to avoid numerical issues near 0 Hz
    lo_n = max(lo_n, 1e-4)
    hi_n = min(hi_n, 1 - 1e-4)
    b, a = butter(order, [lo_n, hi_n], btype='band')
    return filtfilt(b, a, signal, axis=-1)

In [ ]:
# --- Filter bank feature extraction: mean power per band per channel ---
# Result shape: (n_epochs, n_channels, n_bands)
n_bands = len(band_edges)
features = np.zeros((n_epochs, n_channels, n_bands))

for b_idx, (lo, hi) in enumerate(band_edges):
    filtered = bandpass(data, lo, hi, sfreq)            # (n_epochs, n_ch, n_times)
    features[:, :, b_idx] = (filtered ** 2).mean(axis=-1)  # mean band power

print(f"Feature matrix shape: {features.shape}  (epochs x channels x bands)")

In [ ]:
# --- Fisher Ratio: fr = (mu_A - mu_B)^2 / (sigma_A^2 + sigma_B^2) ---
# Compute Fisher Ratio for each class vs. rest (one-vs-rest)
# fr shape: (n_classes, n_channels, n_bands)
fisher_ratios = np.zeros((len(classes), n_channels, n_bands))

for c_idx, code in enumerate(classes):
    mask = labels == code
    X_class = features[mask]        # (n_class, n_channels, n_bands)
    X_rest = features[~mask]        # (n_rest, n_channels, n_bands)
    mu_c = X_class.mean(axis=0)
    mu_r = X_rest.mean(axis=0)
    var_c = X_class.var(axis=0)
    var_r = X_rest.var(axis=0)
    fisher_ratios[c_idx] = (mu_c - mu_r) ** 2 / (var_c + var_r + 1e-12)

print(f"Fisher Ratio shape: {fisher_ratios.shape}")
print(f"Max Fisher Ratio: {fisher_ratios.max():.4f}")

In [ ]:
# --- Visualize: Fisher Ratio heatmap (channels x bands) averaged across classes ---
mean_fr = fisher_ratios.mean(axis=0)   # (n_channels, n_bands)
band_centers = [(lo + hi) / 2 for lo, hi in band_edges]
ch_names = epochs.ch_names

fig, ax = plt.subplots(figsize=(14, 5))
im = ax.imshow(mean_fr, aspect='auto', origin='lower',
               extent=[band_centers[0], band_centers[-1], 0, n_channels])
ax.set_xlabel('Band center frequency (Hz)')
ax.set_ylabel('Channel index')
ax.set_title('Mean Fisher Ratio: channels \u00d7 frequency bands (averaged across classes)')
plt.colorbar(im, ax=ax, label='Fisher Ratio')
plt.tight_layout()
plt.show()

In [ ]:
# --- Top discriminative (channel, band) pairs ---
top_k = 10
flat_idx = np.argsort(mean_fr.ravel())[::-1][:top_k]
ch_idx_top, band_idx_top = np.unravel_index(flat_idx, mean_fr.shape)
print(f"Top {top_k} discriminative (channel, band) pairs:")
for rank, (ci, bi) in enumerate(zip(ch_idx_top, band_idx_top)):
    lo, hi = band_edges[bi]
    print(f"  {rank+1:2d}. Channel {ch_names[ci]:>8}  Band {lo:.0f}\u2013{hi:.0f} Hz  "
          f"Fisher Ratio = {mean_fr[ci, bi]:.4f}")